# Advanced Ingestion Patterns

This notebook covers advanced ingestion scenarios that build on the basics:

1. **Write policy interaction** — how `on_conflict` policies affect ingestion behaviour
2. **Temporal validity** — versioned data with `valid_from` / `valid_to` windows
3. **Reporters** — track ingestion progress and row-level outcomes
4. **Multi-driver ingestion** — writing to PG + Iceberg/DuckDB simultaneously

**Prerequisites**: Familiarity with the basic ingestion notebook (`ingestion_basic`)
and the write policy notebook (`collection_write_policy`).

In [ ]:
# === Parameters ===
import os
BASE = os.environ.get("DYNASTORE_BASE_URL") or "http://localhost:8080"
CATALOG_ID = "demo-advanced-ingestion"

In [ ]:
import httpx
import json

client = httpx.AsyncClient(base_url=BASE, timeout=60)

def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    suffix = "" if ok else f" — {r.text[:300]}"
    print(f"{label + ': ' if label else ''}{status}{suffix}")
    return r

r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Advanced Ingestion Demo", "description": "Advanced Ingestion Demo catalog."})
_check(r, "Catalog")

---
## 1. Write policy + ingestion

The `collection:write_policy` config controls what happens when an ingested
feature collides with an existing entity that shares the same identity.

| `on_conflict` | Behaviour |
|---------------|-----------|
| `update` | Overwrite existing entity (default) |
| `new_version` | Archive old version, insert new one with fresh geoid |
| `refuse` | Skip duplicate, continue processing rest of batch |

The `external_id_field` determines which field is used for identity matching.
When set, conflict resolution uses the external ID instead of the system geoid.

### Interaction with ingestion

The write policy is applied **per upsert batch** during the ingestion loop.
The policy MUST be configured **before** running ingestion.

In [ ]:
COLL_UPDATE = "sensors-update"

# Create collection
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_UPDATE,
    "title": "Sensors (update policy)",
    "description": "Sensors (update policy) collection.",
    "license": "CC-BY-4.0",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r, "Collection")

# Set write policy: update on conflict, track by station_id
policy = {
    "on_conflict": "update",
    "external_id_field": "id",
    "track_asset_id": True
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_UPDATE}/classes/collection:write_policy",
    json=policy,
)
_check(r, "Policy")

# First ingest
feature = {
    "type": "Feature",
    "id": "sensor-001",
    "geometry": {"type": "Point", "coordinates": [12.49, 41.89]},
    "properties": {"temperature": 22.3, "status": "nominal"},
}
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_UPDATE}/items",
    json=feature,
)
_check(r, "First ingest")

# Re-ingest with updated temperature — overwrites
feature["properties"]["temperature"] = 25.1
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_UPDATE}/items",
    json=feature,
)
_check(r, "Re-ingest (update)")

# Verify: temperature should be 25.1
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_UPDATE}/items?limit=10")
items = r.json().get("features", [])
if items:
    print("Stored temperature:", items[0]["properties"]["temperature"])
else:
    print("No items stored")

In [ ]:
COLL_REFUSE = "boundaries-refuse"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_REFUSE, "title": "Boundaries (refuse policy)",
    "description": "Boundaries (refuse policy) collection.",
    "license": "ODbL",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r, "Collection")

# Refuse duplicates — skip silently, continue batch
policy = {"on_conflict": "refuse", "external_id_field": "properties.iso3"}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/classes/collection:write_policy",
    json=policy,
)
_check(r, "Policy")

batch = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [12.5, 41.9]},
     "properties": {"iso3": "ITA", "name": "Italy"}},
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [2.3, 48.9]},
     "properties": {"iso3": "FRA", "name": "France"}},
]}
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/items", json=batch,
)
_check(r, "First batch")

# Re-ingest Italy (refused) + Germany (new)
batch2 = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [12.5, 41.9]},
     "properties": {"iso3": "ITA", "name": "Italy MODIFIED"}},
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [10.4, 51.2]},
     "properties": {"iso3": "DEU", "name": "Germany"}},
]}
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/items", json=batch2,
)
_check(r, "Second batch (ITA refused, DEU written)")

r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/items")
items = r.json().get("features", [])
print(f"Total stored: {len(items)}")
for it in items:
    print(f"  {it['properties']['iso3']}: {it['properties']['name']}")

---
## 2. Temporal validity

When `enable_validity` is set in the write policy, each ingested feature
carries a temporal window (`valid_from` / `valid_to`). Combined with
`on_conflict: new_version`, this creates a full temporal history.

In the ingestion request, specify which source columns hold the temporal values:

```json
{
    "ingestion_request": {
        "time_validity_start_column": "reference_date",
        "time_validity_end_column": null
    }
}
```

The write policy must also enable validity:

```json
{
    "on_conflict": "new_version",
    "enable_validity": true,
    "validity_field": "reference_date",
    "external_id_field": "properties.parcel_id"
}
```

In [ ]:
COLL_TEMPORAL = "land-use-versioned"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_TEMPORAL,
    "title": "Land Use (versioned)",
    "description": "Land Use (versioned) collection.",
    "license": "CC-BY-4.0",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r, "Collection")

# Enable temporal versioning
policy = {
    "on_conflict": "new_version",
    "enable_validity": True,
    "validity_field": "reference_year",
    "external_id_field": "properties.parcel_id",
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_TEMPORAL}/classes/collection:write_policy",
    json=policy,
)
_check(r, "Policy")

# 2022 snapshot
parcel = {
    "type": "Feature",
    "geometry": {"type": "Polygon", "coordinates": [[
        [12.0, 41.0], [12.1, 41.0], [12.1, 41.1], [12.0, 41.1], [12.0, 41.0]
    ]]},
    "properties": {"parcel_id": "P-00123", "land_class": "arable", "reference_year": "2022"},
}
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_TEMPORAL}/items", json=parcel,
)
_check(r, "2022 ingest")

# 2024 reclassification — creates new version, archives 2022
parcel["properties"]["land_class"] = "forest"
parcel["properties"]["reference_year"] = "2024"
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_TEMPORAL}/items", json=parcel,
)
_check(r, "2024 ingest (new version)")

# Both versions are stored
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_TEMPORAL}/items?limit=10")
items = r.json().get("features", [])
print(f"Stored versions: {len(items)}")
for it in items:
    p = it["properties"]
    print(f"  parcel={p.get('parcel_id')} class={p.get('land_class')} year={p.get('reference_year')}")

---
## 3. Reporters

Reporters track ingestion progress and outcomes using the **Strategy Pattern**.
They run alongside the main ingestion loop without polluting the processing code.

### Built-in reporters

| Reporter | Purpose | Output |
|----------|---------|--------|
| `DatabaseStatusReporter` | Updates task `progress` column in DB | Poll via `GET /tasks/catalogs/{cat}/{task_id}` |
| `GcsDetailedReporter` | Row-by-row pass/fail log | JSON file in GCS bucket at task finish |

### Reporter lifecycle

```
task_started(task_id, collection_id, catalog_id, source)
  │
  ├─ update_progress(current, total)  ← called per batch
  ├─ update_progress(current, total)
  │   ...
  └─ task_finished(status="COMPLETED" | "FAILED", error_message=None)
```

### Configuration in the request body

The `reporting` field accepts reporter class names with their config:

```json
{
    "ingestion_request": {
        "reporting": {
            "GcsDetailedReporter": {
                "bucket": "my-catalog-reports",
                "include_geometry": false
            }
        }
    }
}
```

The `DatabaseStatusReporter` is always active (built-in). Additional reporters
are opt-in via the `reporting` config.

---
## 4. Multi-driver ingestion

DynaStore supports writing to multiple storage drivers simultaneously.
The `CollectionRoutingConfig` routing config determines which drivers receive writes.

When you ingest data into a collection with multiple WRITE drivers, the
ingestion engine calls `ensure_storage()` and `upsert()` on **all** WRITE-routed
drivers.

### Common multi-driver patterns

| Pattern | WRITE drivers | READ | SEARCH | Use case |
|---------|--------------|------|--------|----------|
| PG + ES | postgresql | postgresql | elasticsearch | Full-text search over PG data |
| Iceberg + ES | iceberg | iceberg | elasticsearch | Lakehouse analytics + search |
| DuckDB + ES | duckdb | duckdb | elasticsearch | Analytical queries + search |

See the **storage combo notebooks** (`storage_combo_duckdb_plus_es`, etc.)
for detailed routing configuration examples.

In [ ]:
COLL_MULTI = "crop-yield-multi"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_MULTI,
    "title": "Crop Yield (PG + ES)",
    "description": "Crop Yield (PG + ES) collection.",
    "license": "CC-BY-4.0",
    "extent": {"spatial": {"bbox": [[-180, -90, 180, 90]]}, "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]}},
})
_check(r, "Collection")

# Route WRITE to both PG and ES, SEARCH to ES
routing = {
    "operations": {
        "WRITE":  [{"driver_id": "ItemsPostgresqlDriver"}, {"driver_id": "ItemsElasticsearchDriver"}],
        "READ":   [{"driver_id": "ItemsPostgresqlDriver"}],
        "SEARCH": [{"driver_id": "ItemsElasticsearchDriver"}],
    },
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_MULTI}/classes/CollectionRoutingConfig",
    json=routing,
)
_check(r, "Routing")

# Ingest data — lands in both PG and ES
features = {"type": "FeatureCollection", "features": [
    {
        "type": "Feature",
        "id": f"yield-{i:03d}",
        "geometry": {"type": "Point", "coordinates": [10 + i * 0.5, 42 + i * 0.3]},
        "properties": {
            "crop": ["wheat", "corn", "rice", "soybean"][i % 4],
            "yield_t_ha": round(2.0 + i * 0.7, 1),
            "year": 2024,
        },
    }
    for i in range(8)
]}

r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_MULTI}/items", json=features,
)
_check(r, "Ingest")

# Verify via PG (READ)
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_MULTI}/items")
print(f"Items via PG: {r.json().get('numberMatched', 0)}")

# Search via ES (full-text)
r = await client.get(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_MULTI}/items",
    params={"q": "wheat"},
)
print(f"ES search 'wheat': {len(r.json().get('features', []))} results")

In [ ]:
# Cleanup
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
_check(r, "Cleanup")
await client.aclose()